# Content-Based Similarity Engine

## Project Overview

The objective of this notebook is to develop a content-based similarity engine for recipes using Natural Language Processing (NLP).

Unlike the nutrition-based recommendation engine developed previously, this module focuses on learning semantic relationships between recipes by analyzing textual attributes such as recipe names, descriptions, categories, keywords, ingredients, and dietary classifications.

Rather than generating final recommendations independently, the similarity scores produced in this notebook will serve as one component of the Hybrid Recommendation System. These scores will later be combined with nutrition matching, collaborative filtering, and popularity-based metrics to generate personalized recipe recommendations.

By the end of this notebook, each recipe will be represented as a numerical TF-IDF feature vector, enabling efficient computation of semantic similarity between recipes.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Open this notebook from the NutriCore repository or its notebooks directory.")

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
os.chdir(PROJECT_ROOT)

In [ ]:
# Import Libraries

import re

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## Load Processed Dataset

The recipe dataset has already undergone extensive preprocessing in previous notebooks.

The completed preprocessing pipeline includes:

- Feature selection
- Missing value handling
- Dietary classification
- Per-serving nutritional feature engineering
- Outlier removal
- Dataset validation

Since the dataset is already cleaned and standardized, this notebook focuses solely on building the content-based similarity engine.

In [3]:
# Load Processed Dataset

df = pd.read_parquet("data/processed/recipes_processed.parquet")

print(f"Dataset Shape : {df.shape}")

Dataset Shape : (302965, 16)


## Text Features

A content-based recommendation system relies on descriptive information rather than user interactions.

Several textual attributes are available within the processed dataset.

These attributes collectively describe each recipe from different perspectives:

- Recipe name provides the primary identity of the recipe.
- Description summarizes the recipe.
- Recipe category represents the general type of dish.
- Keywords provide descriptive tags assigned to the recipe.
- Ingredient list captures the core components used in the recipe.
- Diet type distinguishes between Vegetarian, Eggitarian, and Non-Vegetarian recipes.

Combining these attributes enables the recommendation system to capture richer semantic relationships between recipes than using any individual feature alone.

In [4]:
# Text Features

TEXT_COLUMNS = [
    "Name",
    "Description",
    "RecipeCategory",
    "Keywords",
    "RecipeIngredientParts",
    "DietType"
]

df[TEXT_COLUMNS].head()

,Name,Description,RecipeCategory,Keywords,RecipeIngredientParts,DietType
0,Low-Fat Berry Blue Frozen Dessert,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Healthy, Free Of..., Summer, Weeknight, Freezer, Easy]","[blueberries, granulated sugar, vanilla yogurt, lemon juice]",Vegetarian
1,Biryani,Make and share this Biryani recipe from Food.com.,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, Asian, Indian, Weeknight, Stove Top]","[saffron, milk, hot green chili peppers, onions, garlic, clove, peppercorns, cardamom seed, cumin seed, poppy seed, mace, cilantro, mint leaf, fresh lemon juice, plain yogurt, boneless chicken, salt, ghee, onion, tomatoes, basmati rice, long-grain rice, raisins, cashews, eggs]",Non-Vegetarian
2,Best Lemonade,"This is from one of my first Good House Keeping cookbooks. You must use a *zester* in order to avoid getting any of that bitter rind, and when you zest the lemons, zest them onto some sugar from the recipe (the sugar will 'catch' all of the oils). I also advise you from personal experience to use only the best skinned lemons for the best flavor.",Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer, < 60 Mins]","[sugar, lemons, rind of, lemon, zest of, fresh water, fresh lemon juice]",Vegetarian
3,Carina's Tofu-Vegetable Kebabs,This dish is best prepared a day in advance to allow the ingredients to soak in the marinade overnight.,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight, Broil/Grill, Oven]","[extra firm tofu, eggplant, zucchini, mushrooms, soy sauce, low sodium soy sauce, olive oil, maple syrup, honey, red wine vinegar, lemon juice, garlic cloves, mustard powder, black pepper]",Vegetarian
4,Cabbage Soup,Make and share this Cabbage Soup recipe from Food.com.,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy, Winter, < 60 Mins, Easy]","[plain tomato juice, cabbage, onion, carrots, celery]",Vegetarian


## Text Preprocessing

Before recipes can be compared using Natural Language Processing (NLP), the textual attributes must be transformed into a consistent format.

The preprocessing performed in this section includes:

- Handling any remaining missing values.
- Converting list-based features into plain text.
- Normalizing text to lowercase.
- Removing unnecessary punctuation.
- Removing common Food.com boilerplate phrases.
- Combining multiple textual attributes into a single feature representation.

Not all attributes contribute equally to describing a recipe. Recipe names and ingredient lists generally provide stronger semantic information than descriptions or categories. To reflect this, selected features are intentionally repeated when constructing the combined text representation. This increases their influence during TF-IDF vectorization without modifying the underlying algorithm.

During dataset inspection, it was observed that many recipe descriptions contained repetitive boilerplate text such as *"Make and share this..."* or simply repeated the recipe name after preprocessing. Such descriptions provide little additional semantic information and unnecessarily duplicate existing features.

To address this, descriptions are retained only when they contain meaningful information beyond the recipe title. This preserves useful cooking tips, serving suggestions, and contextual information while avoiding redundant text.

The resulting feature serves as the foundation for computing semantic similarity between recipes.

In [5]:
# Handle Missing Values

df[TEXT_COLUMNS] = df[TEXT_COLUMNS].fillna("")

In [6]:
# Common Domain-Specific Phrases

COMMON_PHRASES = [
    "make and share this",
    "recipe from food com",
    "recipe from food",
    "food com",
    "food",
    "com"
]


def preprocess_text(text):
    """
    Clean and normalize textual data.
    """

    if isinstance(text, list):
        text = " ".join(text)

    text = str(text).lower()

    # Remove common Food.com boilerplate phrases
    for phrase in COMMON_PHRASES:
        text = text.replace(phrase, " ")

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def create_combined_features(row):
    """
    Combine multiple recipe attributes into a single
    weighted text representation.
    """

    name = preprocess_text(row["Name"])
    description = preprocess_text(row["Description"])

    # Remove descriptions that simply duplicate the recipe name
    if description == name:
        description = ""

    category = preprocess_text(row["RecipeCategory"])
    keywords = preprocess_text(row["Keywords"])
    ingredients = preprocess_text(row["RecipeIngredientParts"])
    diet = preprocess_text(row["DietType"])

    combined_text = " ".join([
        (name + " ") * 3,
        description,
        category,
        (keywords + " ") * 2,
        (ingredients + " ") * 3,
        diet
    ])

    return combined_text.strip()


In [7]:
df["CombinedFeatures"] = df.apply(
    create_combined_features,
    axis=1
)

### Combined Text Representation

Before generating numerical feature vectors, it is important to verify that the combined textual representation has been created correctly.

The inspection below confirms that all selected recipe attributes have been successfully merged into a single document while preserving the intended weighting strategy.

This validation step helps identify preprocessing issues before applying TF-IDF vectorization.

In [8]:
df["CombinedFeatures"].sample(
    10,
    random_state=42
)

94311                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               meatballs meatballs meatballs   meat 30 mins easy 30 mins easy  parmesan cheese milk eggs onions butter ground beef salt pepper garlic powder parmesan cheese milk eggs onions butter ground beef salt pepper garlic powder parmesan cheese milk eggs onions butter ground beef salt

## TF-IDF Representation

Textual recipe information must be converted into a numerical representation before recipe similarity can be computed.

This project uses **Term Frequency–Inverse Document Frequency (TF-IDF)** vectorization, a widely adopted Natural Language Processing (NLP) technique for representing textual documents.

TF-IDF assigns higher importance to words that appear frequently within a specific recipe while reducing the influence of words that are common across many recipes. This enables the recommendation system to focus on distinctive recipe characteristics such as ingredients, cooking styles, and descriptive keywords rather than generic terms.

Additionally, both single words (unigrams) and two-word phrases (bigrams) are considered during vectorization. Capturing common phrases such as *"olive oil"* or *"chicken breast"* provides richer semantic information than treating each word independently.

The resulting sparse matrix forms the numerical representation of every recipe and serves as the foundation for similarity computation.

In [9]:
## TF-IDF Representation

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df["CombinedFeatures"])

print(f"TF-IDF Matrix Shape : {tfidf_matrix.shape}")
print(f"Vocabulary Size    : {len(tfidf.vocabulary_):,}")

TF-IDF Matrix Shape : (302965, 50000)
Vocabulary Size    : 50,000


### TF-IDF Findings

The generated TF-IDF matrix represents each recipe as a high-dimensional sparse feature vector.

Each row corresponds to a recipe, while each column represents a unique unigram or bigram selected from the vocabulary.

Due to the large vocabulary and the fact that each recipe contains only a small subset of all possible terms, the resulting matrix is highly sparse. Sparse matrices are memory-efficient and are commonly used in information retrieval and recommendation systems.

These numerical feature vectors provide a compact representation of each recipe's textual characteristics and enable efficient computation of semantic similarity.

Rather than generating recommendations directly, the similarity scores derived from these vectors will later be incorporated into the Hybrid Recommendation System alongside nutrition matching, collaborative filtering, and popularity-based metrics.

## Similarity Search

The TF-IDF feature matrix provides a numerical representation of every recipe.

A straightforward approach to content-based recommendation is to compute the similarity between every pair of recipes. However, for a dataset containing over 300,000 recipes, constructing a complete pairwise similarity matrix would require an impractical amount of memory and computation.

Instead, this project computes similarity scores only when required. Given a target recipe, its TF-IDF representation is compared against all other recipes to generate similarity scores on demand.

This approach is significantly more memory-efficient and is commonly adopted in large-scale recommendation systems.

The following sections develop the utilities required to efficiently compute semantic similarity between recipes.

### Recipe Index Mapping

Efficient similarity computation requires locating the corresponding TF-IDF vector for a given recipe.

To achieve this, a lookup table is created that maps each unique `RecipeId` to its corresponding row index in the dataset.

Using `RecipeId` provides a stable and unambiguous identifier throughout the recommendation pipeline. This design also aligns with the collaborative filtering and hybrid recommendation system, where recipes are consistently referenced using their unique identifiers.

In [10]:
# Recipe ID to Index Mapping

recipe_id_to_index = pd.Series(
    data=df.index,
    index=df["RecipeId"]
)


### Similarity Computation

Once a recipe has been identified, its TF-IDF representation can be compared with all other recipes in the dataset.

Since TF-IDF vectors are L2-normalized by default, cosine similarity can be computed efficiently using the linear kernel. This produces the same similarity values as cosine similarity while remaining computationally efficient for sparse matrices.

The function below computes semantic similarity scores for a given recipe. These scores form the content-based component of the Hybrid Recommendation System.

In [11]:
# Content Similarity Function

def compute_similarity_scores(recipe_id):
    """
    Compute semantic similarity scores for a given recipe.

    Parameters
    ----------
    recipe_id : int
        Unique Recipe ID.

    Returns
    -------
    numpy.ndarray
        Similarity scores between the selected recipe
        and every recipe in the dataset.
    """

    recipe_id = int(recipe_id)

    if recipe_id not in recipe_id_to_index.index:
        raise ValueError(f"RecipeId '{recipe_id}' not found.")

    recipe_index = recipe_id_to_index.loc[recipe_id]

    similarity_scores = linear_kernel(
        tfidf_matrix[recipe_index],
        tfidf_matrix
    ).flatten()

    return similarity_scores

### Similarity Function Check

Before using similarity scores within the hybrid recommendation system, it is important to verify that the similarity computation behaves as expected.

The following example computes the similarity between a selected recipe and every other recipe in the dataset.

The selected recipe should have a similarity score of **1.0** with itself, while all remaining recipes receive values between **0** and **1**, where higher values indicate greater semantic similarity.

In [12]:
# Example Similarity Scores

recipe_id = 38

similarity_scores = compute_similarity_scores(recipe_id)

print(f"Number of Scores : {len(similarity_scores):,}")
print(f"Maximum Score    : {similarity_scores.max():.4f}")
print(f"Minimum Score    : {similarity_scores.min():.4f}")

Number of Scores : 302,965
Maximum Score    : 1.0000
Minimum Score    : 0.0000


### Similar Recipe Retrieval

The similarity scores computed in the previous section quantify the semantic relationship between the selected recipe and every other recipe in the dataset.

For validation purposes, the helper function below retrieves the recipes with the highest similarity scores.

This function is intended solely for demonstrating and evaluating the effectiveness of the content similarity engine. In the final application, the similarity scores generated by `compute_similarity_scores()` will be combined with nutrition matching, collaborative filtering, and popularity-based metrics within the Hybrid Recommendation System.

In [13]:
# Similar Recipe Retrieval

def recommend_similar_recipes(recipe_name, top_n=10):
    """
    Retrieve the most semantically similar recipes.
    Intended for notebook demonstration only.
    """

    recipe = df[df["Name"] == recipe_name]

    if recipe.empty:
        raise ValueError(f"Recipe '{recipe_name}' not found.")

    recipe_id = recipe.iloc[0]["RecipeId"]

    similarity_scores = compute_similarity_scores(recipe_id)

    similarity_df = pd.DataFrame({
        "RecipeId": df["RecipeId"],
        "Name": df["Name"],
        "RecipeCategory": df["RecipeCategory"],
        "DietType": df["DietType"],
        "SimilarityScore": similarity_scores
    })

    similarity_df = similarity_df[
        similarity_df["RecipeId"] != recipe_id
    ]

    similarity_df = (
        similarity_df
        .sort_values(
            by="SimilarityScore",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return similarity_df

### Similarity Evaluation

The helper function below retrieves the recipes with the highest semantic similarity to a selected recipe.

The retrieved recipes are inspected qualitatively to verify that the content similarity engine captures meaningful relationships based on textual attributes such as ingredients, categories, keywords, and descriptions.

In [14]:
# Example Recommendations

recommend_similar_recipes(
    recipe_name="Low-Fat Berry Blue Frozen Dessert",
    top_n=10
)

,RecipeId,Name,RecipeCategory,DietType,SimilarityScore
0,237886.0,Berry Blue Smoothie,Smoothies,Vegetarian,0.392254
1,517827.0,Blue Berry Cake,Dessert,Vegetarian,0.337153
2,245303.0,Low Fat Peach Frozen Yogurt,Frozen Desserts,Vegetarian,0.332772
3,183601.0,Nectarine-Berry Pie,Pie,Eggitarian,0.316678
4,306162.0,"Red, White, and Blue Jello",Dessert,Vegetarian,0.314873
5,64611.0,Pineapple Yogurt Popsicles,Frozen Desserts,Vegetarian,0.303162
6,311116.0,Berry Yogurt Pops,Frozen Desserts,Vegetarian,0.292263
7,399632.0,Ninja Strawberry Frozen Yogurt,Frozen Desserts,Vegetarian,0.286777
8,375013.0,Blueberry Fool on a Berry Fruit Salad (England),Dessert,Vegetarian,0.284241
9,212717.0,Frozen Vanilla Yogurt Ice Cream (Low Fat),Frozen Desserts,Vegetarian,0.283301


## User Preference Profiles

While recipe-to-recipe similarity is useful for validating the content similarity engine, recommendation systems ultimately operate at the **user level** rather than the **recipe level**.

Instead of relying on a single recipe, a user's content preferences are represented by the collection of recipes they have positively interacted with.

For each liked recipe, semantic similarity scores are computed against every recipe in the dataset. These similarity vectors are then aggregated using the **maximum similarity** to construct a single content preference profile for the user.

This approach ensures that a recipe is recommended if it strongly matches at least one of the user's preferences, rather than requiring it to match all preferences simultaneously. It also prevents score dilution in cases where users have diverse tastes.

The resulting content scores quantify how closely every recipe matches the user's preferences and will later be integrated into the Hybrid Recommendation System.

In [15]:
# User Content Profile

def compute_content_scores(
    liked_recipe_ids,
    return_details=False
):
    """
    Compute aggregated content similarity scores for a user.

    Parameters
    ----------
    liked_recipe_ids : list
        List of positively rated RecipeIds.

    return_details : bool, default=False
        If True, return recipe information along with
        the computed content scores.

    Returns
    -------
    pandas.DataFrame
        If return_details=False:
            RecipeId, ContentScore

        If return_details=True:
            RecipeId, Name, RecipeCategory,
            DietType, ContentScore
    """

    if len(liked_recipe_ids) == 0:
        raise ValueError(
            "At least one liked recipe is required."
        )

    similarity_matrix = []

    for recipe_id in liked_recipe_ids:
        similarity_matrix.append(
            compute_similarity_scores(recipe_id)
        )

    similarity_matrix = np.vstack(similarity_matrix)

    content_scores = similarity_matrix.max(axis=0)

    content_df = pd.DataFrame({
        "RecipeId": df["RecipeId"],
        "ContentScore": content_scores
    })

    # Remove recipes already liked
    content_df.loc[
        content_df["RecipeId"].isin(liked_recipe_ids),
        "ContentScore"
    ] = 0

    content_df = (
        content_df
        .sort_values(
            by="ContentScore",
            ascending=False
        )
        .reset_index(drop=True)
    )

    if not return_details:
        return content_df

    content_df = content_df.merge(
        df[
            [
                "RecipeId",
                "Name",
                "RecipeCategory",
                "DietType"
            ]
        ],
        on="RecipeId",
        how="left"
    )

    return content_df[
        [
            "RecipeId",
            "Name",
            "RecipeCategory",
            "DietType",
            "ContentScore"
        ]
    ]

### User Content Profile Check

The following example demonstrates how multiple liked recipes are combined to generate a unified content preference profile.

Recipes that the user has already liked are excluded from the final ranking to avoid recommending items the user has previously interacted with.

### User Preference Inspection

Before generating recommendations, it is useful to examine the recipes that define the user's content profile.

The recipes below represent the user's positively rated interactions and serve as the basis for computing aggregated content similarity scores.

This inspection helps verify that the generated recommendations align with the user's historical preferences.

In [16]:
# Liked Recipes

liked_recipe_ids = [
    38,
    245303,
    517827
]

df[
    df["RecipeId"].isin(liked_recipe_ids)
][
    [
        "RecipeId",
        "Name",
        "RecipeCategory",
        "DietType"
    ]
].reset_index(drop=True)

,RecipeId,Name,RecipeCategory,DietType
0,38.0,Low-Fat Berry Blue Frozen Dessert,Frozen Desserts,Vegetarian
1,245303.0,Low Fat Peach Frozen Yogurt,Frozen Desserts,Vegetarian
2,517827.0,Blue Berry Cake,Dessert,Vegetarian


In [17]:
compute_content_scores(
    liked_recipe_ids,
    return_details=True
).head(10)

,RecipeId,Name,RecipeCategory,DietType,ContentScore
0,325709.0,Just Peachy Yogurt,Breakfast,Vegetarian,0.639019
1,168184.0,My Low Calorie French Toast Made With Yogurt,Breads,Vegetarian,0.586340
2,36585.0,Quick As A wink Smoothie,Smoothies,Vegetarian,0.585028
3,100156.0,Angel Food Cake Dessert,Dessert,Vegetarian,0.550380
4,399632.0,Ninja Strawberry Frozen Yogurt,Frozen Desserts,Vegetarian,0.539594
5,433436.0,Nick's Smoothie,Smoothies,Vegetarian,0.533414
6,39302.0,Peach Yogurt Smoothie,Smoothies,Vegetarian,0.525730
7,244734.0,Fruity Bear Dip,Low Cholesterol,Vegetarian,0.518083
8,179318.0,Cherry Vanilla Cobbler,Dessert,Vegetarian,0.515607
9,413219.0,Bran Pancakes,Breakfast,Vegetarian,0.510523


### Load User Review Data

The content similarity engine is evaluated using the Food.com review dataset.

Each review represents an interaction between a user and a recipe. For this evaluation, recipes with ratings greater than or equal to four stars are considered positive interactions and are used to construct the user's content preference profile.

In [18]:
# Load User Reviews

reviews_df = pd.read_parquet("data/processed/reviews_processed.parquet")

print(f"Reviews Shape : {reviews_df.shape}")

Reviews Shape : (788843, 8)


### Content-Based Recommendations

To evaluate the recommendation engine under realistic conditions, the Food.com reviews dataset is used to simulate an existing user's preferences.

Recipes rated **4 stars or higher** are treated as positive interactions and are used to construct the user's content preference profile.

Similarity scores are then aggregated across all positively rated recipes, and previously rated recipes are removed before generating recommendations.

This evaluation closely resembles how the content-based component of the Hybrid Recommendation System will operate for existing users.

In [19]:
# Existing User Recommendations

def recommend_for_user(
    user_id,
    reviews_df,
    min_rating=4,
    top_n=10,
    show_liked_recipes=False
):
    """
    Generate content-based recommendations for an existing user.

    Parameters
    ----------
    user_id : int
        User identifier.

    reviews_df : pandas.DataFrame
        Processed reviews dataset.

    min_rating : int, default=4
        Minimum rating considered as a positive interaction.

    top_n : int, default=10
        Number of recommendations to return.

    show_liked_recipes : bool, default=False
        If True, also return the recipes used to build
        the user's content profile.

    Returns
    -------
    pandas.DataFrame or tuple
        If show_liked_recipes=False:
            Top-N recommended recipes.

        If show_liked_recipes=True:
            (liked_recipes, recommendations)
    """

    liked_recipe_ids = (
        reviews_df.loc[
            (reviews_df["AuthorId"] == user_id)
            & (reviews_df["Rating"] >= min_rating),
            "RecipeId"
        ]
        .unique()
        .tolist()
    )

    if len(liked_recipe_ids) == 0:
        raise ValueError(
            f"User {user_id} has no recipes rated {min_rating} or higher."
        )

    recommendations = (
        compute_content_scores(
            liked_recipe_ids,
            return_details=True
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    if not show_liked_recipes:
        return recommendations

    liked_recipes = (
        df[
            df["RecipeId"].isin(liked_recipe_ids)
        ][
            [
                "RecipeId",
                "Name",
                "RecipeCategory",
                "DietType"
            ]
        ]
        .reset_index(drop=True)
    )

    return liked_recipes, recommendations

In [20]:
liked_recipes, recommendations = recommend_for_user(
    user_id=20084,
    reviews_df=reviews_df,
    show_liked_recipes=True
)

print("Liked Recipes")
display(liked_recipes)

print("\nRecommended Recipes")
display(recommendations)

Liked Recipes


,RecipeId,Name,RecipeCategory,DietType
0,674.0,Curried Carrot Bisque,Vegetable,Vegetarian
1,2901.0,Orange Roughy with Veggies,Orange Roughy,Eggitarian
2,8626.0,Sandy's Famous Green Beans,Beans,Vegetarian
3,9012.0,Quick Quesadillas,Lunch/Snacks,Vegetarian
4,9972.0,Joan's Simple Lemon Chicken,Chicken,Non-Vegetarian
5,10284.0,Breaded Pork Cutlets With Honey Dijon Mustard,Pork,Non-Vegetarian
6,10343.0,Apple Cabbage Skillet Dinner,One Dish Meal,Non-Vegetarian
7,10406.0,"EASY! Penne with chicken, spinach and tomato Alfredo (soooo good!)",Chicken Breast,Non-Vegetarian
8,12404.0,Lime Cumin Chicken,Chicken Breast,Non-Vegetarian
9,12458.0,Crock Pot Cream Cheese Chicken,One Dish Meal,Non-Vegetarian



Recommended Recipes


,RecipeId,Name,RecipeCategory,DietType,ContentScore
0,89573.0,Balsamic Strawberries,Dessert,Vegetarian,0.754837
1,26698.0,Quick Quesadillas,Cheese,Vegetarian,0.721936
2,519029.0,Chicken Lazone,Chicken,Non-Vegetarian,0.685269
3,374418.0,Balsamic Strawberries,Dessert,Vegetarian,0.675280
4,26405.0,Strawberries with Balsamic,Dessert,Vegetarian,0.660050
5,456654.0,Balsamic Strawberries,Dessert,Vegetarian,0.638665
6,61248.0,Corn-On-The-Cob-Packet,Corn,Vegetarian,0.582937
7,424096.0,Pork Scaloppine With Honey Mustard,Pork,Non-Vegetarian,0.565163
8,456436.0,Balsamic Strawberries - Just a Little Bit Different!,Dessert,Vegetarian,0.551664
9,444891.0,Baked Corn,Corn,Eggitarian,0.551163


## Save Model Artifacts

The trained TF-IDF vectorizer and the corresponding sparse feature matrix are saved for reuse in later stages of the project.

Persisting these artifacts eliminates the need to repeat text preprocessing and vectorization, enabling faster inference during deployment and seamless integration with the Hybrid Recommendation System.

The saved artifacts will be loaded in subsequent notebooks to compute content similarity scores without retraining the vectorizer.

In [21]:
# Save Model Artifacts

import os
import joblib
from scipy.sparse import save_npz

os.makedirs("models", exist_ok=True)

joblib.dump(
    tfidf,
    "models/tfidf_vectorizer.pkl"
)

save_npz(
    "models/tfidf_matrix.npz",
    tfidf_matrix
)

print("Artifacts saved successfully.")

Artifacts saved successfully.


### Artifact Verification

The following cell verifies that the serialized artifacts have been successfully written to disk and can be loaded for future use.

In [22]:
# Verify Saved Artifacts

import joblib
from scipy.sparse import load_npz

# Load vectorizer
tfidf_loaded = joblib.load(
    "models/tfidf_vectorizer.pkl"
)

# Load TF-IDF matrix
tfidf_matrix_loaded = load_npz(
    "models/tfidf_matrix.npz"
)

print(type(tfidf_loaded))
print(type(tfidf_matrix_loaded))
print(tfidf_matrix_loaded.shape)

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
<class 'scipy.sparse._csr.csr_matrix'>
(302965, 50000)


## Conclusion

In this notebook, a content-based similarity engine was successfully developed using textual recipe information extracted from the Food.com dataset.

Key textual attributes including recipe names, categories, keywords, ingredients, and dietary classifications were preprocessed and combined into a unified feature representation. These textual features were transformed into high-dimensional numerical vectors using TF-IDF vectorization, enabling efficient semantic comparison between recipes.

Rather than constructing a computationally expensive pairwise similarity matrix, recipe similarities are computed on demand using cosine similarity via the linear kernel. This approach is scalable and better suited for large recipe collections.

The notebook also demonstrated how similarity scores from multiple positively rated recipes can be aggregated to construct a personalized user content profile. Recommendations generated for both individual recipes and existing Food.com users illustrate the effectiveness of the content similarity engine in capturing semantic relationships between recipes.

The saved TF-IDF artifacts produced in this notebook will be reused in subsequent stages of the project. Specifically, the content similarity scores generated here will later be integrated with collaborative filtering, nutrition-based filtering, and popularity-based ranking to form the final Hybrid Recipe Recommendation System.